In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

# A simple Backtracking Constraint Solver

In [ ]:
type Variable = string;
type Value = string | number;
type Formula = string;
type Assignment = Record<Variable, Value>;
type CSPTuple = [Variable[], Value[], Formula[]];
type AnnotatedCSP = [Variable[], Value[], Array<[Formula, Set<Variable>]>];

The function `collectVariables(expr)` takes a string `expr` that can be interpreted as a valid expression as its input and collects all variables occurring in `expr`.
It takes care to remove names that correspond to predefined values or functions. To achieve this, it defines a set of `builtIns` containing reserved names (like `Math`, `true`, `false`, `abs`, `min`) that are not treated as variables, ensuring that only true variable names are returned by `collectVariables`.

In [ ]:
function collectVariables(expr: string): Set<string> {
    const identifierRegex = /[a-zA-Z_][a-zA-Z0-9_]*/g;
    const builtIns = new Set(['abs', 'min', 'max', 'pow', 'sum', 'len', 'Math', 'true', 'false']);
    const variables = new Set<string>();
    let match: RegExpExecArray | null;
    while ((match = identifierRegex.exec(expr)) !== null) {
        if (!builtIns.has(match[0])) variables.add(match[0]);
    }
    return variables;
}

In [ ]:
const expr = 'abs(x - y) + abs(z1 - z2)';
collectVariables(expr);

The function `isConsistent` takes four arguments:
- `variable` is a variable.
- `value` is a value that is to be assigned to the variable `variable`.
- `assignment` is a partial variable assignment that does not assign a value for `variable`
  and that is *consistent* with all constraints in the set `constraints`.
- `constraints` is a set of annotated logical formulas.

The function checks whether the assignment
$$ \texttt{assignment} \cup \{\texttt{variable} \mapsto \texttt{value}\}$$
violates any of the formulas in `constraints`. It assumes that
`assignment` is *consistent*.

In [ ]:
function isConsistent(
    variable: Variable,
    value: Value,
    assignment: Assignment,
    constraints: Array<[Formula, Set<Variable>]>
): boolean {
    const newAssignment = { ...assignment, [variable]: value };
    const assignedVars = new Set(Object.keys(newAssignment));
    for (const [formula, varsInFormula] of constraints) {
        if (varsInFormula.has(variable) && isSubset(varsInFormula, assignedVars)) {
            try {
                const argNames = Object.keys(newAssignment);
                const argValues = Object.values(newAssignment);
                const checkFunc = new Function(...argNames, `return (${formula});`);
                if (!checkFunc(...argValues)) {
                    return false;
                }
            } catch (e) {
                console.error(`Fehler beim Auswerten von "${formula}":`, e);
                return false; 
            }
        }
    }
    return true;
}
function isSubset<T>(subset: Set<T>, superset: Set<T>): boolean {
    for (let elem of subset) {
        if (!superset.has(elem)) return false;
    }
    return true;
}

Given a **consistent** *partial variable assignment* `assignment` and a constraint satisfaction problem `csp`,
the function `backtrackSearch(assignment, csp)` tries to extend the given assignment recursively and thereby produce a solution of the given CSP.

In [ ]:
function backtrackSearch(assignment: Assignment, csp: AnnotatedCSP): Assignment | null {
    const [variables, values, constraints] = csp;
    if (Object.keys(assignment).length === variables.length) {
        return assignment;
    }
    const unassignedVar = variables.find(v => !(v in assignment));
    for (const value of values) {
        if (isConsistent(unassignedVar, value, assignment, constraints)) {
            const newAssignment = { ...assignment, [unassignedVar]: value };
            const result = backtrackSearch(newAssignment, csp);
            if (result !== null) {
                return result;
            }
        }
    }
    return null;
}

The input to the function `solve` is a *constraint satisfaction problem*, i.e. a *CSP*.
The function `solve` tries to compute a solution of this problem via *backtracking*.
Its main purpose is to transform the given *CSP* into an *annotated CSP* where all the formulas
are *annotated* with their variables. That is, the third component of the *CSP* is now no longer a set of formulas but rather a set of pairs of the form `(f, V)` where `f` is a formula and `V` is the set of variables occurring in this formula. The function then calls the auxiliary function `backtrackSearch` that recursively solves the *annotated CSP*.

In [ ]:
function solve(csp: CSPTuple): Assignment | null {
    const [variables, values, constraints] = csp;
    const annotatedConstraints: Array<[Formula, Set<Variable>]> = constraints.map(f => {
        const varsInFormula = collectVariables(f);
        const varsInProblem = new Set(variables);
        return [f, intersection(varsInFormula, varsInProblem)];
    });

    const annotatedCSP: AnnotatedCSP = [variables, values, annotatedConstraints];
    return backtrackSearch({}, annotatedCSP);
}
function intersection<T>(setA: Set<T>, setB: Set<T>): Set<T> {
    return new Set([...setA].filter(x => setB.has(x)));
}

The helper function `evaluateExpression` takes a string expression and a
variable assignment (context), creates a dynamic function, and evaluates the given expression using the provided values.

In [ ]:
function evaluateExpression(expression: string, context: Record<string, any>): any {
    const argNames = Object.keys(context);
    const argValues = Object.values(context);
    const dynamicFunc = new Function(...argNames, `return (${expression});`);
    return dynamicFunc(...argValues);
}

In [ ]:
const expr = 'x + y == 7 * z';
const context = { 'x': 3, 'y': 4, 'z': 1 };
const result = evaluateExpression(expr, context);
console.log(`Ergebnis für "${expr}":`, result); 